# Memory System — V2: MongoDB + pgvector + Neo4j

A V1 do Memory System usa backends locais: `LongTermMemory` (JSON em disco), `VectorMemory` (Chroma/SQLite local) e `KnowledgeGraphBuilder` (dicionários em memória). Esta V2 troca cada um por um banco de dados real, mantendo a mesma interface pública -- o resto do agente (`MemoryManager`, agentes, RAG) não precisa mudar.

## Imports

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))


## 1. Long-Term Memory — MongoDB

Mesma interface da V1 (`store`, `retrieve`, `search_by_category`, `all_entries`, `delete`), agora persistida em MongoDB com índice único em `key` e índice em `category`.

In [ ]:
from memory.long_term_memory_mongo_v2 import MongoLongTermMemory

ltm = MongoLongTermMemory(uri=os.getenv("MONGODB_URI", "mongodb://localhost:27017"))

ltm.store("user_preference", "respostas curtas e diretas", category="profile")
ltm.store("last_project", "OmniMind AI OS - Memory System V2", category="context")

print(ltm.retrieve("user_preference"))
print(ltm.search_by_category("context"))


## 2. Vector Memory — Postgres + pgvector

Mesma interface da V1 (`store`, `search`, `clear`), com os mesmos embeddings (`all-MiniLM-L6-v2`), agora persistidos em uma tabela Postgres com índice HNSW (`vector_cosine_ops`) para busca aproximada por similaridade.

In [ ]:
from memory.vector_memory_pgvector_v2 import PgVectorMemory

vm = PgVectorMemory(dsn=os.getenv("POSTGRES_DSN", "postgresql://localhost:5432/agent_os"))

vm.store("O usuário prefere respostas curtas e objetivas.", metadata={"source": "chat"})
vm.store("O projeto atual é o OmniMind AI OS.", metadata={"source": "chat"})

vm.search("como o usuário gosta de receber respostas?", k=2)


## 3. Knowledge Graph — Neo4j

Mesma extração de entidades/relações via LLM da V1 (`EntityExtractor`, reaproveitado sem duplicação), agora persistida como nós/relações reais em Neo4j, com `shortestPath` resolvido em Cypher no próprio banco em vez de um BFS manual em Python.

In [ ]:
from knowledge_graph.graph_builder_neo4j_v2 import Neo4jKnowledgeGraphBuilder, Neo4jGraphRetriever

kg = Neo4jKnowledgeGraphBuilder(
    uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "neo4j")),
)

kg.add_text("Marie Curie trabalhou com Pierre Curie na Sorbonne.", source="bio.txt")
print(kg.stats())

retriever = Neo4jGraphRetriever(kg)
print(retriever.get_neighbors("Marie Curie"))
print(retriever.find_path("Marie Curie", "Sorbonne"))


## 4. MemoryManagerV2 — os três juntos

Mesma orquestração da V1 (`short` + `long` + `vector`), agora sobre MongoDB + pgvector.

In [ ]:
from memory.memory_manager_v2 import MemoryManagerV2

manager = MemoryManagerV2(
    mongo_uri=os.getenv("MONGODB_URI"),
    postgres_dsn=os.getenv("POSTGRES_DSN"),
)

manager.add_turn("Qual projeto estamos trabalhando?", "OmniMind AI OS, Memory System V2.")
manager.remember("user_name", "Yuri")

print(manager.build_context_for_agent("qual é o projeto atual?"))
print(manager.recall("user_name"))

manager.close()


## Conclusão

V1 e V2 expõem a mesma interface pública em cada componente de memória -- trocar o backend (JSON → MongoDB, Chroma → pgvector, dict em memória → Neo4j) não exige mudar nenhum agente que já consome `MemoryManager`, `LongTermMemory`, `VectorMemory` ou `KnowledgeGraphBuilder`.